In [1]:
# --- Path setup ---
import sys
import os
import math
import time
import contextlib
import io
import warnings
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "max_k_cut").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

# --- Scientific / data ---
import networkx as nx
import numpy as np
import scipy.io

# --- Visualization ---
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams["figure.dpi"] = 1000
mpl.rcParams["savefig.dpi"] = 1000

# --- Qiskit ecosystem ---
import qiskit
import qiskit_aer
import qiskit_algorithms
import qiskit_optimization
from qiskit.circuit import Parameter
from qiskit import QuantumCircuit
from qiskit.primitives import Sampler, BackendSampler
from qiskit_aer.primitives import Sampler as AerSampler
from qiskit.circuit.library import QAOAAnsatz, RYGate, XGate, CXGate
from qiskit.visualization import plot_histogram, plot_state_city, plot_state_qsphere, plot_bloch_multivector, plot_distribution
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeBrisbane
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer.noise import NoiseModel



from qiskit_algorithms import QAOA, SamplingVQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA

from qiskit_optimization.algorithms import MinimumEigenOptimizer, SolutionSample, OptimizationResultStatus
from qiskit_optimization.problems import QuadraticProgram
from qiskit_optimization.converters import LinearEqualityToPenalty, LinearInequalityToPenalty, QuadraticProgramToQubo
from qiskit_optimization.translators import from_docplex_mp

# --- IPython ---
from IPython.display import display, Math

# --- Project ---
from max_k_cut import (
    docplex_BQO, docplex_RBQO, docplex_QUBO, docplex_RQUBO, docplex_QUBO_no_constraints,
    tight_qubo_penalty, tight_rqubo_penalty, naive_qubo_penalty, naive_rqubo_penalty,
    interpolated_qubo_penalty, interpolated_rqubo_penalty,
    generate_graph, plot_graph, feasibility_filter, expected_value, sample_std,
    hamming_distance_to_feasibility,
    create_dicke_initial_state, create_reduced_dicke_initial_state,
    create_full_xy_mixer, create_ring_xy_mixer,
)
from max_k_cut.qaoa import run_qaoa_extract_samples
from scripts.plot_histogram import plot_qaoa_histogram

warnings.filterwarnings('ignore', category=DeprecationWarning)

# --- Version info ---
print("qiskit version:", qiskit.__version__)
print("qiskit aer version:", qiskit_aer.__version__)
print("qiskit algorithms version:", qiskit_algorithms.__version__)
print("qiskit optimization version:", qiskit_optimization.__version__)

# %config InlineBackend.figure_format = 'retina'

qiskit version: 1.4.6
qiskit aer version: 0.15.1
qiskit algorithms version: 0.3.1
qiskit optimization version: 0.6.1


In [2]:
# --- One-time IBM Quantum account setup ---
# Credentials are written to ~/.qiskit/qiskit-ibm.json (outside the repo) and persist across
# kernels and venv rebuilds. The token is prompted for via getpass and passed straight to
# save_account, so it never lands in this notebook's source, outputs, or a kernel variable.
from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

ACCOUNT_NAME = "QCOL"
CHANNEL = "ibm_quantum_platform"  # requires qiskit-ibm-runtime >= 0.40
OVERWRITE = False  # set True to force re-entering credentials

_saved = QiskitRuntimeService.saved_accounts().get(ACCOUNT_NAME)
_stale = _saved is not None and _saved.get("channel") != CHANNEL

if _saved is not None and not _stale and not OVERWRITE:
    print(f"Account {ACCOUNT_NAME!r} already saved on channel {CHANNEL!r} - skipping.")
else:
    if _stale:
        print(f"Re-saving {ACCOUNT_NAME!r}: stored channel {_saved.get('channel')!r} -> {CHANNEL!r}")
    # Reuse the stored CRN if there is one; press Enter to accept it.
    _default_instance = (_saved or {}).get("instance")
    _prompt = f"Instance (CRN) [{_default_instance}]: " if _default_instance else "Instance (CRN): "
    _instance = input(_prompt).strip() or _default_instance

    # .strip() guards against a stray newline/space picked up when pasting the key.
    QiskitRuntimeService.save_account(
        channel=CHANNEL,
        token=getpass("IBM Quantum API token: ").strip(),
        instance=_instance,
        name=ACCOUNT_NAME,
        set_as_default=True,
        overwrite=True,
    )
    print(f"Saved account {ACCOUNT_NAME!r} to ~/.qiskit/qiskit-ibm.json")

# Sanity check (prints names/channels only, never tokens). Constructing the service verifies
# the key against IBM Cloud IAM, so a bad token surfaces here rather than deeper in the run.
print({k: v.get("channel") for k, v in QiskitRuntimeService.saved_accounts().items()})


Account 'QCOL' already saved on channel 'ibm_quantum_platform' - skipping.
{'QCOL': 'ibm_quantum_platform', 'QLSAs': 'ibm_cloud'}


In [3]:
service = QiskitRuntimeService(name="QCOL")
service.backends()

[<IBMBackend('ibm_pittsburgh')>,
 <IBMBackend('ibm_boston')>,
 <IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_miami')>,
 <IBMBackend('ibm_marrakesh')>,
 <IBMBackend('ibm_kingston')>]

In [4]:
def add_a_weighted_edge(graph, weight):

    # Get all edge data
    edges = list(graph.edges(data=True))

    # Extract edge weights and check if graph is weighted
    weights = [d['weight'] for (_, _, d) in edges]

    # pick a random edge to add weight to
    random_edge = np.random.choice(len(weights))

    # Check if graph is weighted
    (u, v, _) = edges[random_edge]
    graph.edges[u, v]['weight'] = weight

    # Add weight
    # weights[random_edge] = weight

    return graph


test_graph = generate_graph(6, 0.5, False)
test_graph_weighted = add_a_weighted_edge(test_graph, 10)
# plot_graph(test_graph)
test_graph.edges(data=True)

EdgeDataView([(0, 2, {'weight': 1}), (0, 4, {'weight': 1}), (0, 5, {'weight': 1}), (1, 2, {'weight': 1}), (1, 4, {'weight': 1}), (1, 5, {'weight': 1}), (2, 3, {'weight': 1}), (4, 5, {'weight': 10})])

# Create QAOA models for enforcing constraints using penalties, mixers, and a combination of the two

In [5]:
# Experiment parameters
K = 3 # hamming weight constraint
num_nodes = 6 # number of nodes in the graph
edge_probability = 0.5
weighted = False
weight_range = 1
optimizer = COBYLA()
reps = 1
initial_point = np.random.rand(2 * reps) * np.pi / 2 # initial point for the optimizer


cvar_alpha = 0.25
shots = int(5000 / cvar_alpha)

noisy = True

if noisy:
    backend = service.backend("ibm_boston")
    noise_model = NoiseModel.from_backend(backend)
    # Native Aer sampler with noise
    sampler = AerSampler(
            backend_options=dict(noise_model=noise_model),
            run_options=dict(shots=shots)
        )
else:
    noiseless_simulator = AerSimulator()
    sampler = AerSampler(run_options=dict(shots=shots))

In [6]:
# ============================Penalty QAOA============================
penalty_qaoa = QAOA(
    sampler=sampler, 
    optimizer=optimizer, 
    reps=reps, 
    initial_point=initial_point,
    aggregation=cvar_alpha,
    )

# create minimum eigen optimizer based on solver used
penalty_qaoa_optimizer = MinimumEigenOptimizer(penalty_qaoa)

In [7]:
# ============================Dicke-State Penalty QAOA============================
# use with qubo model for equality constraints
init_qc = create_dicke_initial_state(num_nodes, K)

dicke_penalty_qaoa = QAOA(
    sampler=sampler, 
    optimizer=optimizer, 
    reps=reps,
    initial_state=init_qc,
    initial_point=initial_point,
    aggregation=cvar_alpha,
    )

# create minimum eigen optimizer based on solver used
dicke_penalty_qaoa_optimizer = MinimumEigenOptimizer(dicke_penalty_qaoa)
#init_qc.draw(output='mpl')

# ============================Reduced-Dicke Penalty QAOA============================
# use with rqubo model: uniform superposition over each node's 0-hot and 1-hot states
reduced_init_qc = create_reduced_dicke_initial_state(num_nodes, K)

dicke_rqubo_penalty_qaoa = QAOA(
    sampler=sampler, 
    optimizer=optimizer, 
    reps=reps,
    initial_state=reduced_init_qc,
    initial_point=initial_point,
    aggregation=cvar_alpha,
    )

dicke_rqubo_penalty_qaoa_optimizer = MinimumEigenOptimizer(dicke_rqubo_penalty_qaoa)


In [8]:
# ============================Dicke-State Mixer QAOA============================
from qiskit.circuit import Parameter

init_qc = create_dicke_initial_state(num_nodes, K)
beta = Parameter("β")
mixer = create_ring_xy_mixer(num_nodes, K, beta)
# mixer = create_full_xy_mixer(num_nodes, K, beta)

mixer_qaoa = QAOA(
    sampler=sampler, 
    optimizer=optimizer, 
    reps=reps,
    initial_state=init_qc,
    mixer=mixer,
    initial_point=initial_point,
    aggregation=cvar_alpha,
    )

# create minimum eigen optimizer based on solver used
mixer_qaoa_optimizer = MinimumEigenOptimizer(mixer_qaoa)

# XY mixer without CVaR aggregation (standard expectation value)
mixer_qaoa_no_agg = QAOA(
    sampler=sampler,
    optimizer=optimizer,
    reps=reps,
    initial_state=init_qc,
    mixer=mixer,
    initial_point=initial_point,
    )
mixer_qaoa_no_agg_optimizer = MinimumEigenOptimizer(mixer_qaoa_no_agg)

mixer.draw(output='mpl', fold=-1)

In [9]:
# ============================Classical Solver============================
np_solver = NumPyMinimumEigensolver() # exact classical solver
np_optimizer = MinimumEigenOptimizer(np_solver)

In [ ]:
import pickle

# penalty_list = np.linspace(0, 1, 6)
# num_graphs = 20

np.random.seed(42)  # reproducible graph set (and weighted-edge choice)

penalty_list = np.linspace(0, 1, 3)
num_graphs = 10

RESULTS_NPZ = os.path.join("Data", "Noisy_Results_P1_random_init_10_graphs_cvar_violations_6nodes.npz")
RAW_SAMPLES_PKL = os.path.join("Data", "raw_samples_Noisy_P1_random_init_10_graphs_cvar_6nodes.pkl")

QUBO_EXP_VALS     = np.zeros((num_graphs, len(penalty_list)))
QUBO_FEASIBILITY  = np.zeros((num_graphs, len(penalty_list)))
RQUBO_EXP_VALS    = np.zeros((num_graphs, len(penalty_list)))
RQUBO_FEASIBILITY = np.zeros((num_graphs, len(penalty_list)))

DICKE_QUBO_EXP_VALS     = np.zeros((num_graphs, len(penalty_list)))
DICKE_QUBO_FEASIBILITY  = np.zeros((num_graphs, len(penalty_list)))
DICKE_RQUBO_EXP_VALS    = np.zeros((num_graphs, len(penalty_list)))
DICKE_RQUBO_FEASIBILITY = np.zeros((num_graphs, len(penalty_list)))
PENALTY_MIXER_QUBO_EXP_VALS     = np.zeros((num_graphs, len(penalty_list)))
PENALTY_MIXER_QUBO_FEASIBILITY  = np.zeros((num_graphs, len(penalty_list)))

XY_MIXER_EXP_VALS = np.zeros(num_graphs)
XY_MIXER_FEASIBILITY = np.zeros(num_graphs)
XY_MIXER_NO_AGG_EXP_VALS = np.zeros(num_graphs)
XY_MIXER_NO_AGG_FEASIBILITY = np.zeros(num_graphs)

# --- Constraint-violation (Hamming distance to feasibility) accumulators ---
# Common pmf support 0..n(K-1) for all encodings (RQUBO capped at n(K-2) by construction).
MAX_DIST = num_nodes * (K - 1)
QUBO_VIOLATION_PMF               = np.zeros((num_graphs, len(penalty_list), MAX_DIST + 1))
RQUBO_VIOLATION_PMF              = np.zeros((num_graphs, len(penalty_list), MAX_DIST + 1))
DICKE_QUBO_VIOLATION_PMF         = np.zeros((num_graphs, len(penalty_list), MAX_DIST + 1))
DICKE_RQUBO_VIOLATION_PMF        = np.zeros((num_graphs, len(penalty_list), MAX_DIST + 1))
PENALTY_MIXER_QUBO_VIOLATION_PMF = np.zeros((num_graphs, len(penalty_list), MAX_DIST + 1))
XY_MIXER_VIOLATION_PMF           = np.zeros((num_graphs, MAX_DIST + 1))
XY_MIXER_NO_AGG_VIOLATION_PMF    = np.zeros((num_graphs, MAX_DIST + 1))

# E[distance | infeasible]; NaN when a run produced only feasible samples
QUBO_EXP_VIOLATION_INFEAS               = np.full((num_graphs, len(penalty_list)), np.nan)
RQUBO_EXP_VIOLATION_INFEAS              = np.full((num_graphs, len(penalty_list)), np.nan)
DICKE_QUBO_EXP_VIOLATION_INFEAS         = np.full((num_graphs, len(penalty_list)), np.nan)
DICKE_RQUBO_EXP_VIOLATION_INFEAS        = np.full((num_graphs, len(penalty_list)), np.nan)
PENALTY_MIXER_QUBO_EXP_VIOLATION_INFEAS = np.full((num_graphs, len(penalty_list)), np.nan)
XY_MIXER_EXP_VIOLATION_INFEAS           = np.full(num_graphs, np.nan)
XY_MIXER_NO_AGG_EXP_VIOLATION_INFEAS    = np.full(num_graphs, np.nan)

# Raw per-shot data, persisted so future metrics don't require re-running the simulation.
# Keyed by (graph_index, model_name, pen_index); pen_index is None for the XY mixers.
raw_samples = {}

def pack_samples(samples):
    return {
        "x": np.stack([np.asarray(s.x) for s in samples]).astype(np.uint8),
        "fval": np.array([s.fval for s in samples], dtype=float),
        "prob": np.array([s.probability for s in samples], dtype=float),
    }

for graph in range(num_graphs):
    # generate random graph
    G = generate_graph(num_nodes, edge_probability, weighted, weight_range)
    # add a random edge with weight 100
    G = add_a_weighted_edge(G, 10)
    raw_samples[(graph, "graph_edges", None)] = list(G.edges(data=True))

    with contextlib.redirect_stdout(io.StringIO()):
        dp_qubo_no_constraints = docplex_QUBO_no_constraints(G, K, "Max-K-Cut")
        mixer_results = run_qaoa_extract_samples(
            [dp_qubo_no_constraints],
            ["XY Mixer"],
            optimizer=mixer_qaoa_optimizer
        )

    xy_samples = mixer_results["XY Mixer"]["samples"]
    raw_samples[(graph, "XY Mixer", None)] = pack_samples(xy_samples)
    viol_xy = hamming_distance_to_feasibility(G, K, xy_samples, "QUBO (XY Mixer)")
    xy_filtered, xy_feas_prob = feasibility_filter(G, K, xy_samples, "QUBO (XY Mixer)")
    assert np.isclose(viol_xy["feasibility_probability"], xy_feas_prob)
    xy_exp_val = expected_value(xy_filtered)
    XY_MIXER_EXP_VALS[graph] = xy_exp_val
    XY_MIXER_FEASIBILITY[graph] = xy_feas_prob
    XY_MIXER_VIOLATION_PMF[graph] = viol_xy["pmf"]
    XY_MIXER_EXP_VIOLATION_INFEAS[graph] = viol_xy["expected_distance_infeasible"]

    with contextlib.redirect_stdout(io.StringIO()):
        dp_qubo_no_constraints_no_agg = docplex_QUBO_no_constraints(G, K, "Max-K-Cut")
        mixer_no_agg_results = run_qaoa_extract_samples(
            [dp_qubo_no_constraints_no_agg],
            ["XY Mixer (No Agg)"],
            optimizer=mixer_qaoa_no_agg_optimizer
        )

    xy_no_agg_samples = mixer_no_agg_results["XY Mixer (No Agg)"]["samples"]
    raw_samples[(graph, "XY Mixer (No Agg)", None)] = pack_samples(xy_no_agg_samples)
    viol_xy_no_agg = hamming_distance_to_feasibility(G, K, xy_no_agg_samples, "QUBO (XY Mixer)")
    xy_no_agg_filtered, xy_no_agg_feas_prob = feasibility_filter(G, K, xy_no_agg_samples, "QUBO (XY Mixer)")
    assert np.isclose(viol_xy_no_agg["feasibility_probability"], xy_no_agg_feas_prob)
    xy_no_agg_exp_val = expected_value(xy_no_agg_filtered)
    XY_MIXER_NO_AGG_EXP_VALS[graph] = xy_no_agg_exp_val
    XY_MIXER_NO_AGG_FEASIBILITY[graph] = xy_no_agg_feas_prob
    XY_MIXER_NO_AGG_VIOLATION_PMF[graph] = viol_xy_no_agg["pmf"]
    XY_MIXER_NO_AGG_EXP_VIOLATION_INFEAS[graph] = viol_xy_no_agg["expected_distance_infeasible"]

    print(f"Graph {graph+1}/{num_graphs} XY Mixer:")
    print(f"  With CVaR: Expected Value: {xy_exp_val:.4f}, Feasibility Probability: {xy_feas_prob:.4f}")
    print(f"  No Agg:    Expected Value: {xy_no_agg_exp_val:.4f}, Feasibility Probability: {xy_no_agg_feas_prob:.4f}")

    for pen_index, pen in enumerate(penalty_list):
        # Create docplex models
        dp_qubo_interp = docplex_QUBO(G, K, interpolated_qubo_penalty(G, K, pen), "Max-K-Cut")
        dp_rqubo_interp = docplex_RQUBO(G, K, interpolated_rqubo_penalty(G, K, pen), "Max-K-Cut")

        with contextlib.redirect_stdout(io.StringIO()):
            # Run QAOA
            penalty_results = run_qaoa_extract_samples(
                [dp_qubo_interp, dp_rqubo_interp],
                ["QUBO (Interpolated)", "RQUBO (Interpolated)"],
                optimizer=penalty_qaoa_optimizer
            )
            dicke_results = run_qaoa_extract_samples(
                [dp_qubo_interp],
                ["QUBO (Interpolated)"],
                optimizer=dicke_penalty_qaoa_optimizer
            )
            dicke_rqubo_results = run_qaoa_extract_samples(
                [dp_rqubo_interp],
                ["RQUBO (Interpolated)"],
                optimizer=dicke_rqubo_penalty_qaoa_optimizer
            )
            penalty_mixer_results = run_qaoa_extract_samples(
                [dp_qubo_interp],
                ["QUBO (Interpolated)"],
                optimizer=mixer_qaoa_optimizer
            )

        # Retrieve the raw samples
        samples_qubo = penalty_results["QUBO (Interpolated)"]["samples"]
        samples_rqubo = penalty_results["RQUBO (Interpolated)"]["samples"]
        samples_dicke_qubo = dicke_results["QUBO (Interpolated)"]["samples"]
        samples_dicke_rqubo = dicke_rqubo_results["RQUBO (Interpolated)"]["samples"]
        samples_penalty_mixer_qubo = penalty_mixer_results["QUBO (Interpolated)"]["samples"]

        raw_samples[(graph, "QUBO", pen_index)] = pack_samples(samples_qubo)
        raw_samples[(graph, "RQUBO", pen_index)] = pack_samples(samples_rqubo)
        raw_samples[(graph, "Dicke QUBO", pen_index)] = pack_samples(samples_dicke_qubo)
        raw_samples[(graph, "Dicke RQUBO", pen_index)] = pack_samples(samples_dicke_rqubo)
        raw_samples[(graph, "Penalty+Mixer QUBO", pen_index)] = pack_samples(samples_penalty_mixer_qubo)

        # Constraint-violation distributions (on raw samples, before filtering renormalizes)
        viol_qubo = hamming_distance_to_feasibility(G, K, samples_qubo, "QUBO (Interpolated)")
        viol_rqubo = hamming_distance_to_feasibility(G, K, samples_rqubo, "RQUBO (Interpolated)")
        viol_dicke = hamming_distance_to_feasibility(G, K, samples_dicke_qubo, "QUBO (Interpolated)")
        viol_dicke_rqubo = hamming_distance_to_feasibility(G, K, samples_dicke_rqubo, "RQUBO (Interpolated)")
        viol_penalty_mixer = hamming_distance_to_feasibility(G, K, samples_penalty_mixer_qubo, "QUBO (Interpolated)")

        # Filter the samples and get the feasibility probability.
        filtered_samples_qubo, feas_prob_qubo = feasibility_filter(G, K, samples_qubo, "QUBO (Interpolated)")
        filtered_samples_rqubo, feas_prob_rqubo = feasibility_filter(G, K, samples_rqubo, "RQUBO (Interpolated)")
        filtered_samples_dicke, feas_prob_dicke = feasibility_filter(G, K, samples_dicke_qubo, "QUBO (Interpolated)")
        filtered_samples_dicke_rqubo, feas_prob_dicke_rqubo = feasibility_filter(G, K, samples_dicke_rqubo, "RQUBO (Interpolated)")
        filtered_samples_penalty_mixer, feas_prob_penalty_mixer = feasibility_filter(G, K, samples_penalty_mixer_qubo, "QUBO (Interpolated)")

        # Consistency: distance 0 <=> feasible
        assert np.isclose(viol_qubo["feasibility_probability"], feas_prob_qubo)
        assert np.isclose(viol_rqubo["feasibility_probability"], feas_prob_rqubo)
        assert np.isclose(viol_dicke["feasibility_probability"], feas_prob_dicke)
        assert np.isclose(viol_dicke_rqubo["feasibility_probability"], feas_prob_dicke_rqubo)
        assert np.isclose(viol_penalty_mixer["feasibility_probability"], feas_prob_penalty_mixer)

        # Calculate the expected value of the objective function.
        exp_val_qubo = expected_value(filtered_samples_qubo)
        exp_val_rqubo = expected_value(filtered_samples_rqubo)
        exp_val_dicke = expected_value(filtered_samples_dicke)
        exp_val_dicke_rqubo = expected_value(filtered_samples_dicke_rqubo)
        exp_val_penalty_mixer = expected_value(filtered_samples_penalty_mixer)

        # Store the results
        QUBO_EXP_VALS[graph, pen_index] = exp_val_qubo
        QUBO_FEASIBILITY[graph, pen_index] = feas_prob_qubo
        QUBO_VIOLATION_PMF[graph, pen_index] = viol_qubo["pmf"]
        QUBO_EXP_VIOLATION_INFEAS[graph, pen_index] = viol_qubo["expected_distance_infeasible"]

        RQUBO_EXP_VALS[graph, pen_index] = exp_val_rqubo
        RQUBO_FEASIBILITY[graph, pen_index] = feas_prob_rqubo
        RQUBO_VIOLATION_PMF[graph, pen_index] = viol_rqubo["pmf"]
        RQUBO_EXP_VIOLATION_INFEAS[graph, pen_index] = viol_rqubo["expected_distance_infeasible"]

        DICKE_QUBO_EXP_VALS[graph, pen_index] = exp_val_dicke
        DICKE_QUBO_FEASIBILITY[graph, pen_index] = feas_prob_dicke
        DICKE_QUBO_VIOLATION_PMF[graph, pen_index] = viol_dicke["pmf"]
        DICKE_QUBO_EXP_VIOLATION_INFEAS[graph, pen_index] = viol_dicke["expected_distance_infeasible"]

        DICKE_RQUBO_EXP_VALS[graph, pen_index] = exp_val_dicke_rqubo
        DICKE_RQUBO_FEASIBILITY[graph, pen_index] = feas_prob_dicke_rqubo
        DICKE_RQUBO_VIOLATION_PMF[graph, pen_index] = viol_dicke_rqubo["pmf"]
        DICKE_RQUBO_EXP_VIOLATION_INFEAS[graph, pen_index] = viol_dicke_rqubo["expected_distance_infeasible"]

        PENALTY_MIXER_QUBO_EXP_VALS[graph, pen_index] = exp_val_penalty_mixer
        PENALTY_MIXER_QUBO_FEASIBILITY[graph, pen_index] = feas_prob_penalty_mixer
        PENALTY_MIXER_QUBO_VIOLATION_PMF[graph, pen_index] = viol_penalty_mixer["pmf"]
        PENALTY_MIXER_QUBO_EXP_VIOLATION_INFEAS[graph, pen_index] = viol_penalty_mixer["expected_distance_infeasible"]

        # Print the results
        print(f"Graph {graph+1}/{num_graphs}, Penalty {pen:.2f}:")
        print(f"  QUBO Expected Value: {exp_val_qubo:.4f}, Feasibility Probability: {feas_prob_qubo:.4f}")
        print(f"  RQUBO Expected Value: {exp_val_rqubo:.4f}, Feasibility Probability: {feas_prob_rqubo:.4f}")
        print(f"  Dicke QUBO Expected Value: {exp_val_dicke:.4f}, Feasibility Probability: {feas_prob_dicke:.4f}")
        print(f"  Dicke RQUBO Expected Value: {exp_val_dicke_rqubo:.4f}, Feasibility Probability: {feas_prob_dicke_rqubo:.4f}")
        print(f"  Penalty+Mixer QUBO Expected Value: {exp_val_penalty_mixer:.4f}, Feasibility Probability: {feas_prob_penalty_mixer:.4f}")

# save results (new filename: original published npz is left untouched)
os.makedirs("Data", exist_ok=True)
np.savez(
    RESULTS_NPZ,
    QUBO_EXP_VALS=QUBO_EXP_VALS,
    QUBO_FEASIBILITY=QUBO_FEASIBILITY,
    RQUBO_EXP_VALS=RQUBO_EXP_VALS,
    RQUBO_FEASIBILITY=RQUBO_FEASIBILITY,
    DICKE_QUBO_EXP_VALS=DICKE_QUBO_EXP_VALS,
    DICKE_QUBO_FEASIBILITY=DICKE_QUBO_FEASIBILITY,
    DICKE_RQUBO_EXP_VALS=DICKE_RQUBO_EXP_VALS,
    DICKE_RQUBO_FEASIBILITY=DICKE_RQUBO_FEASIBILITY,
    PENALTY_MIXER_QUBO_EXP_VALS=PENALTY_MIXER_QUBO_EXP_VALS,
    PENALTY_MIXER_QUBO_FEASIBILITY=PENALTY_MIXER_QUBO_FEASIBILITY,
    XY_MIXER_EXP_VALS=XY_MIXER_EXP_VALS,
    XY_MIXER_FEASIBILITY=XY_MIXER_FEASIBILITY,
    XY_MIXER_NO_AGG_EXP_VALS=XY_MIXER_NO_AGG_EXP_VALS,
    XY_MIXER_NO_AGG_FEASIBILITY=XY_MIXER_NO_AGG_FEASIBILITY,
    QUBO_VIOLATION_PMF=QUBO_VIOLATION_PMF,
    RQUBO_VIOLATION_PMF=RQUBO_VIOLATION_PMF,
    DICKE_QUBO_VIOLATION_PMF=DICKE_QUBO_VIOLATION_PMF,
    DICKE_RQUBO_VIOLATION_PMF=DICKE_RQUBO_VIOLATION_PMF,
    PENALTY_MIXER_QUBO_VIOLATION_PMF=PENALTY_MIXER_QUBO_VIOLATION_PMF,
    XY_MIXER_VIOLATION_PMF=XY_MIXER_VIOLATION_PMF,
    XY_MIXER_NO_AGG_VIOLATION_PMF=XY_MIXER_NO_AGG_VIOLATION_PMF,
    QUBO_EXP_VIOLATION_INFEAS=QUBO_EXP_VIOLATION_INFEAS,
    RQUBO_EXP_VIOLATION_INFEAS=RQUBO_EXP_VIOLATION_INFEAS,
    DICKE_QUBO_EXP_VIOLATION_INFEAS=DICKE_QUBO_EXP_VIOLATION_INFEAS,
    DICKE_RQUBO_EXP_VIOLATION_INFEAS=DICKE_RQUBO_EXP_VIOLATION_INFEAS,
    PENALTY_MIXER_QUBO_EXP_VIOLATION_INFEAS=PENALTY_MIXER_QUBO_EXP_VIOLATION_INFEAS,
    XY_MIXER_EXP_VIOLATION_INFEAS=XY_MIXER_EXP_VIOLATION_INFEAS,
    XY_MIXER_NO_AGG_EXP_VIOLATION_INFEAS=XY_MIXER_NO_AGG_EXP_VIOLATION_INFEAS,
    penalty_list=penalty_list,
)

# persist raw per-shot samples so future metrics don't require re-running the simulation
with open(RAW_SAMPLES_PKL, "wb") as f:
    pickle.dump(raw_samples, f)
print(f"Saved results to {RESULTS_NPZ} and raw samples to {RAW_SAMPLES_PKL}")

Probability of feasible solution for QUBO (XY Mixer): 0.7001
Probability of feasible solution for QUBO (XY Mixer): 0.7032
Graph 1/10 XY Mixer:
  With CVaR: Expected Value: 0.6812, Feasibility Probability: 0.7001
  No Agg:    Expected Value: 0.7039, Feasibility Probability: 0.7032
Probability of feasible solution for QUBO (Interpolated): 0.0460
Probability of feasible solution for RQUBO (Interpolated): 0.2147
Probability of feasible solution for QUBO (Interpolated): 0.7147
Probability of feasible solution for QUBO (Interpolated): 0.6577
Graph 1/10, Penalty 0.00:
  QUBO Expected Value: 0.7048, Feasibility Probability: 0.0460
  RQUBO Expected Value: 0.6974, Feasibility Probability: 0.2147
  Dicke QUBO Expected Value: 0.6676, Feasibility Probability: 0.7147
  Penalty+Mixer QUBO Expected Value: 0.6736, Feasibility Probability: 0.6577
Probability of feasible solution for QUBO (Interpolated): 0.0044
Probability of feasible solution for RQUBO (Interpolated): 0.2434
Probability of feasible solu

# Backfill: add Dicke RQUBO to an existing run without re-simulating the other models

Run the imports cell and the experiment-parameters cell first, then this cell — skip the main experiment cell above.

In [10]:
# --- Backfill: run ONLY the Dicke RQUBO variant on the graph set of an existing run ---
# Reconstructs the exact graphs from raw_samples[(g, "graph_edges", None)] in an existing
# raw-samples pickle, runs Dicke RQUBO per (graph, penalty), and merges DICKE_RQUBO_* into
# the existing npz. All other models' stored results are untouched, so the new model is
# directly comparable to the already-simulated data. Re-running reuses stored Dicke RQUBO
# samples from the pickle unless FORCE_RERUN_BACKFILL = True.
# Requires the imports cell and the experiment-parameters cell (sampler/optimizer/reps/
# initial_point/cvar_alpha/K); does NOT require the main experiment cell above.

import pickle

BACKFILL_NPZ = os.path.join("Data", "Noisy_Results_P1_random_init_10_graphs_cvar_violations.npz")
BACKFILL_PKL = os.path.join("Data", "raw_samples_Noisy_P1_random_init_10_graphs_cvar.pkl")
FORCE_RERUN_BACKFILL = False

with open(BACKFILL_PKL, "rb") as f:
    bf_raw = pickle.load(f)
bf_data = dict(np.load(BACKFILL_NPZ))

bf_penalty_list = bf_data["penalty_list"]
bf_num_graphs = bf_data["QUBO_EXP_VALS"].shape[0]
# Node count of the stored run (may differ from the parameter cell): RQUBO register width = n*(K-1)
bf_num_nodes = bf_raw[(0, "RQUBO", 0)]["x"].shape[1] // (K - 1)
bf_max_dist = bf_num_nodes * (K - 1)
assert bf_data["QUBO_VIOLATION_PMF"].shape[2] == bf_max_dist + 1


def bf_graph_from_edges(edge_list, n):
    Gr = nx.Graph()
    Gr.add_nodes_from(range(n))
    for u, v, attrs in edge_list:
        Gr.add_edge(u, v, weight=attrs["weight"])
    return Gr


def bf_pack(samples):
    return {
        "x": np.stack([np.asarray(s.x) for s in samples]).astype(np.uint8),
        "fval": np.array([s.fval for s in samples], dtype=float),
        "prob": np.array([s.probability for s in samples], dtype=float),
    }


def bf_unpack(packed):
    return [
        SolutionSample(x=np.asarray(x, dtype=float), fval=float(f), probability=float(p),
                       status=OptimizationResultStatus.SUCCESS)
        for x, f, p in zip(packed["x"], packed["fval"], packed["prob"])
    ]


# Optimizer sized for the stored run's node count
bf_qaoa = QAOA(
    sampler=sampler,
    optimizer=optimizer,
    reps=reps,
    initial_state=create_reduced_dicke_initial_state(bf_num_nodes, K),
    initial_point=initial_point,
    aggregation=cvar_alpha,
)
bf_optimizer = MinimumEigenOptimizer(bf_qaoa)

DICKE_RQUBO_EXP_VALS = np.zeros((bf_num_graphs, len(bf_penalty_list)))
DICKE_RQUBO_FEASIBILITY = np.zeros((bf_num_graphs, len(bf_penalty_list)))
DICKE_RQUBO_VIOLATION_PMF = np.zeros((bf_num_graphs, len(bf_penalty_list), bf_max_dist + 1))
DICKE_RQUBO_EXP_VIOLATION_INFEAS = np.full((bf_num_graphs, len(bf_penalty_list)), np.nan)

for graph in range(bf_num_graphs):
    G_bf = bf_graph_from_edges(bf_raw[(graph, "graph_edges", None)], bf_num_nodes)
    for pen_index, pen in enumerate(bf_penalty_list):
        key = (graph, "Dicke RQUBO", pen_index)
        if key in bf_raw and not FORCE_RERUN_BACKFILL:
            samples_bf = bf_unpack(bf_raw[key])
        else:
            with contextlib.redirect_stdout(io.StringIO()):
                dp_bf = docplex_RQUBO(G_bf, K, interpolated_rqubo_penalty(G_bf, K, pen), "Max-K-Cut")
                bf_results = run_qaoa_extract_samples(
                    [dp_bf], ["RQUBO (Interpolated)"], optimizer=bf_optimizer
                )
            samples_bf = bf_results["RQUBO (Interpolated)"]["samples"]
            bf_raw[key] = bf_pack(samples_bf)

        viol_bf = hamming_distance_to_feasibility(G_bf, K, samples_bf, "RQUBO (Interpolated)")
        with contextlib.redirect_stdout(io.StringIO()):
            filtered_bf, feas_bf = feasibility_filter(G_bf, K, samples_bf, "RQUBO (Interpolated)")
        assert np.isclose(viol_bf["feasibility_probability"], feas_bf)

        DICKE_RQUBO_EXP_VALS[graph, pen_index] = expected_value(filtered_bf)
        DICKE_RQUBO_FEASIBILITY[graph, pen_index] = feas_bf
        DICKE_RQUBO_VIOLATION_PMF[graph, pen_index] = viol_bf["pmf"]
        DICKE_RQUBO_EXP_VIOLATION_INFEAS[graph, pen_index] = viol_bf["expected_distance_infeasible"]
        print(f"Backfill graph {graph+1}/{bf_num_graphs}, t={pen:.2f}: "
              f"E[ratio|feas]={DICKE_RQUBO_EXP_VALS[graph, pen_index]:.4f}, P_feas={feas_bf:.4f}")

# Merge into the npz (existing keys preserved) and persist the new raw samples
bf_data["DICKE_RQUBO_EXP_VALS"] = DICKE_RQUBO_EXP_VALS
bf_data["DICKE_RQUBO_FEASIBILITY"] = DICKE_RQUBO_FEASIBILITY
bf_data["DICKE_RQUBO_VIOLATION_PMF"] = DICKE_RQUBO_VIOLATION_PMF
bf_data["DICKE_RQUBO_EXP_VIOLATION_INFEAS"] = DICKE_RQUBO_EXP_VIOLATION_INFEAS
np.savez(BACKFILL_NPZ, **bf_data)
with open(BACKFILL_PKL, "wb") as f:
    pickle.dump(bf_raw, f)
print(f"Merged DICKE_RQUBO_* into {BACKFILL_NPZ} and updated {BACKFILL_PKL}")


Backfill graph 1/10, t=0.00: E[ratio|feas]=0.7360, P_feas=0.6869
Backfill graph 1/10, t=0.50: E[ratio|feas]=0.6786, P_feas=0.6762
Backfill graph 1/10, t=1.00: E[ratio|feas]=0.6926, P_feas=0.7923
Backfill graph 2/10, t=0.00: E[ratio|feas]=0.8443, P_feas=0.7609
Backfill graph 2/10, t=0.50: E[ratio|feas]=0.6442, P_feas=0.8388
Backfill graph 2/10, t=1.00: E[ratio|feas]=0.6675, P_feas=0.8814
Backfill graph 3/10, t=0.00: E[ratio|feas]=0.8674, P_feas=0.7660
Backfill graph 3/10, t=0.50: E[ratio|feas]=0.6968, P_feas=0.9305
Backfill graph 3/10, t=1.00: E[ratio|feas]=0.7583, P_feas=0.8778
Backfill graph 4/10, t=0.00: E[ratio|feas]=0.8756, P_feas=0.6834
Backfill graph 4/10, t=0.50: E[ratio|feas]=0.7641, P_feas=0.9057
Backfill graph 4/10, t=1.00: E[ratio|feas]=0.6785, P_feas=0.9315
Backfill graph 5/10, t=0.00: E[ratio|feas]=0.8033, P_feas=0.7812
Backfill graph 5/10, t=0.50: E[ratio|feas]=0.7441, P_feas=0.7961
Backfill graph 5/10, t=1.00: E[ratio|feas]=0.6879, P_feas=0.8051
Backfill graph 6/10, t=0.

# goal with this run: set P=1 (down from 4), increase graph size to 6 (up from 4)

QUBO tight        395.8s
QUBO naive        395.1s
RQUBO tight       28.9s
RQUBO naive       30.1s
QUBO (XY)         1732.1s
QUBO tight        optimum at (γ=0.094, β=2.673)  value=10.1068  (classical=17.0000)
QUBO naive        optimum at (γ=6.258, β=0.390)  value=-11.0099  (classical=17.0000)
RQUBO tight       optimum at (γ=0.063, β=2.764)  value=11.8389  (classical=17.0000)
RQUBO naive       optimum at (γ=0.050, β=2.726)  value=11.2664  (classical=17.0000)
QUBO (XY)         optimum at (γ=0.170, β=3.035)  value=14.8585  (classical=18.0000)